In [1]:
# def _study_records(batch_dir):
#     result_root = Path(batch_dir) / "module06_architecture_experiments"
#     if not result_root.is_dir():
#         return []
#     records = []
#     for validation_path in result_root.rglob("*_optuna_validation_metrics.csv"):
#         if "trials" in validation_path.relative_to(result_root).parts:
#             continue
#         try:
#             validation = pd.read_csv(validation_path)
#         except Exception:
#             continue
#         if validation.empty:
#             continue
#         architecture = (
#             str(validation["Architecture"].dropna().iloc[0])
#             if "Architecture" in validation.columns and validation["Architecture"].notna().any()
#             else validation_path.parent.name
#         )
#         variants = (
#             sorted(validation["Variant"].dropna().astype(str).unique().tolist())
#             if "Variant" in validation.columns else []
#         )
#         records.append({
#             "batch_dir": Path(batch_dir),
#             "result_root": result_root,
#             "study_key": validation_path.parent.name,
#             "architecture": architecture,
#             "variants": variants,
#             "validation_path": validation_path,
#             "study_dir": validation_path.parent,
#         })
#     return records


# def _load_optuna_study(record):
#     database_path = record["study_dir"] / "optuna" / "optuna_studies.sqlite3"
#     if not database_path.is_file():
#         return None
#     storage = f"sqlite:///{database_path.resolve().as_posix()}"
#     try:
#         return optuna.load_study(
#             study_name=f"module06_{record['study_key']}",
#             storage=storage,
#         )
#     except Exception:
#         return None


# def _matches_model_name(record, requested_name):
#     requested = str(requested_name).strip().casefold()
#     architecture = record["architecture"]
#     exact_options = {
#         architecture.casefold(),
#         record["study_key"].casefold(),
#         f"{architecture} :: {'; '.join(record['variants'])}".casefold(),
#     }
#     if architecture == "CrossAttention_ThreeModalities":
#         exact_options.add("CrossAttention_ThreeModalities_VariantWindow".casefold())
#     elif architecture == "CrossAttention_ThreeModalities_InputScope":
#         for variant in record["variants"]:
#             if variant.startswith("center_token"):
#                 exact_options.add("CrossAttention_ThreeModalities_CenterToken".casefold())
#             elif variant.startswith("full_tokens_unpooled"):
#                 exact_options.add("CrossAttention_ThreeModalities_FullTokens".casefold())
#     return requested in exact_options


# def _find_records_for_model(model_name):
#     if BATCH_RUN_DIR_OVERRIDE is not None:
#         batch = Path(BATCH_RUN_DIR_OVERRIDE)
#         if not batch.is_absolute():
#             batch = REPO_ROOT / batch
#         candidate_batches = [batch.resolve()]
#     else:
#         candidate_batches = sorted(
#             (path for path in EXPERIMENTS_ROOT.glob("module06_architecture_run_*")
#              if path.is_dir()),
#             key=lambda path: path.name,
#             reverse=True,
#         )

#     for batch in candidate_batches:
#         matches = [
#             record for record in _study_records(batch)
#             if _matches_model_name(record, model_name)
#         ]
#         completed = []
#         for record in matches:
#             study = _load_optuna_study(record)
#             if study is not None and study.best_trials:
#                 record["study"] = study
#                 completed.append(record)
#         if completed:
#             return batch, completed
#     return None, []


# def _checkpoint_row(target_config, checkpoint_dir):
#     best_model = checkpoint_dir / "best_model.pth"
#     training_state = checkpoint_dir / "training_state.pth"
#     return {
#         "Target_Config": target_config,
#         "Best model checkpoint": str(best_model),
#         "best_model.pth exists": best_model.is_file(),
#         "Best model size (MB)": (
#             round(best_model.stat().st_size / (1024 ** 2), 2)
#             if best_model.is_file() else None
#         ),
#         "Resume state": str(training_state),
#         "training_state.pth exists": training_state.is_file(),
#     }


# MODEL_INSPECTION = {}
# for requested_name in MODEL_NAMES_TO_INSPECT:
#     batch_dir, records = _find_records_for_model(requested_name)
#     if not records:
#         raise FileNotFoundError(
#             f"No completed Optuna study found for model name {requested_name!r} "
#             f"under {EXPERIMENTS_ROOT}."
#         )

#     print(f"\nRequested model: {requested_name}")
#     print(f"Batch selected automatically: {batch_dir}")
#     print("Matching completed studies:")
#     display(pd.DataFrame([
#         {
#             "Architecture": record["architecture"],
#             "Study_Key": record["study_key"],
#             "Variants": "; ".join(record["variants"]),
#         }
#         for record in records
#     ]))

#     for record in records:
#         study = record["study"]
#         best_trial = study.best_trial
#         params = dict(best_trial.params)
#         params_json = json.dumps(params, sort_keys=True, separators=(",", ":"))
#         artifact_key = "params_" + hashlib.sha1(
#             params_json.encode("utf-8")
#         ).hexdigest()[:12]
#         trial_dir = record["study_dir"] / "trials" / artifact_key

#         print(
#             f"\n{record['architecture']} | {record['study_key']} | "
#             f"Optuna trial {best_trial.number} | "
#             f"mean validation MCC A/B = {best_trial.value:.6f}"
#         )
#         print("Optuna-selected hyperparameters:")
#         display(pd.DataFrame([
#             {"Hyperparameter": key, "Best value": value}
#             for key, value in params.items()
#         ]))

#         manifest_path = trial_dir / "_resume_manifest.json"
#         full_config = None
#         if manifest_path.is_file():
#             manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
#             full_config = manifest.get("config", {}).get(
#                 f"{record['study_key']}_config"
#             )
#         if full_config is not None:
#             print("Full model/training configuration:")
#             display(pd.DataFrame([
#                 {"Setting": key, "Value": value}
#                 for key, value in full_config.items()
#             ]))
#         else:
#             print(f"Full configuration manifest not found: {manifest_path}")

#         checkpoint_rows = [
#             _checkpoint_row(
#                 target_config,
#                 trial_dir / "checkpoints" / target_config,
#             )
#             for target_config in ("A_no_geometry", "B_with_geometry")
#         ]
#         print("Best checkpoints saved for this trial:")
#         display(pd.DataFrame(checkpoint_rows))

#         validation_path = trial_dir / f"{record['study_key']}_validation_metrics.csv"
#         validation_result = None
#         if validation_path.is_file():
#             validation_result = pd.read_csv(validation_path)
#             columns = [
#                 column for column in (
#                     "Target_Config", "Best_Epoch", "Decision_Threshold",
#                     "Val_MCC", "Val_AUROC", "Val_AUPRC",
#                 )
#                 if column in validation_result.columns
#             ]
#             print("Validation checkpoint and threshold details:")
#             display(validation_result[columns])

#         MODEL_INSPECTION[(record["study_key"], best_trial.number)] = {
#             "requested_model_name": requested_name,
#             "architecture": record["architecture"],
#             "batch_dir": str(batch_dir),
#             "best_trial_number": best_trial.number,
#             "objective_value": best_trial.value,
#             "params": params,
#             "full_config": full_config,
#             "trial_dir": str(trial_dir),
#             "checkpoints": checkpoint_rows,
#         }


# Notebook 14 ? FM ablations and model checkpoints

Quy tr?nh FM ablation ???c ??t tr??c ph?n ki?m tra checkpoint. Ch?y c?c cell theo th? t?: preflight/cache inventory ? resume token extraction cho b??c 1. K?t qu? b??c 1 v? 2 s? quy?t ??nh model/sequence length cho c?c b??c sau. **Pooling extraction c?a b??c 3 kh?ng ch?y ? giai ?o?n ??u.** Ph?n checkpoint inspector ? cu?i v?n ch? ??c k?t qu? ?? l?u.


## FM ablation ? preflight v? c?u h?nh
Cell n?y khai b?o 7 FM, s?u split (train3, val v? b?n t?p test), v? ki?m tra cache tr??c khi extraction. `RESUME` ch? b? sung cache thi?u ho?c ch?a ho?n ch?nh.


In [2]:
from pathlib import Path
from datetime import datetime
import os, sys, json
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "core").is_dir() and (p / "controllers").is_dir()), _cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from core.module06_architecture_experiments.fm_ablation_preparation import (
    BASE_MODELS, build_model_configs, build_split_configs,
    inspect_token_cache, extract_missing_token_caches,
)

# Keep variant inputs and FM caches on F; repository experiment outputs stay under REPO_ROOT on D.
BASE_DIR = Path(r"F:\variant_data")
POOLED_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "sequence_ablation"
TOKEN_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "token_level_cache" / "module06"
# D?ng l?i batch run hi?n h?u n?u c?n resume; n?u ?? None, t?o batch ri?ng cho FM ablation.
RESUME_FM_ABLATION_BATCH = True
FM_ABLATION_BATCH_RUN_DIR = r"experiments\fm_ablation_20261117_003149"  # Optional explicit path overrides the pointer.
FM_ABLATION_BATCH_POINTER = REPO_ROOT / "experiments" / "_fm_ablation_latest_batch.json"
if FM_ABLATION_BATCH_RUN_DIR is not None:
    FM_ABLATION_BATCH_RUN_DIR = Path(FM_ABLATION_BATCH_RUN_DIR)
    if not FM_ABLATION_BATCH_RUN_DIR.is_absolute():
        FM_ABLATION_BATCH_RUN_DIR = REPO_ROOT / FM_ABLATION_BATCH_RUN_DIR
elif RESUME_FM_ABLATION_BATCH and FM_ABLATION_BATCH_POINTER.is_file():
    _batch_pointer = json.loads(FM_ABLATION_BATCH_POINTER.read_text(encoding="utf-8"))
    FM_ABLATION_BATCH_RUN_DIR = Path(_batch_pointer["batch_run_dir"]).resolve()
else:
    FM_ABLATION_BATCH_RUN_DIR = REPO_ROOT / "experiments" / f"fm_ablation_{datetime.now():%Y%m%d_%H%M%S}"
    FM_ABLATION_BATCH_POINTER.parent.mkdir(parents=True, exist_ok=True)
    FM_ABLATION_BATCH_POINTER.write_text(
        json.dumps({"batch_run_dir": str(FM_ABLATION_BATCH_RUN_DIR.resolve())}, indent=2),
        encoding="utf-8",
    )

# B??c 1: DNA 1001 ? Protein 501; 3 DNA model ? 4 protein model = 12 t? h?p.
STEP1_TOKEN_MODELS = build_model_configs(dna_length=1001, protein_length=501)
STEP1_SPLITS = build_split_configs(BASE_DIR)
print("FM ablation batch:", FM_ABLATION_BATCH_RUN_DIR)
print("Token cache root:", TOKEN_CACHE_ROOT)
print("Models:", [m["name"] for m in STEP1_TOKEN_MODELS])
print("Splits:", [s["name"] for s in STEP1_SPLITS])


FM ablation batch: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149
Token cache root: F:\variant_data\fm_embeddings\token_level_cache\module06
Models: ['nt_v1_500m_dna_1001', 'nt_v3_650m_dna_1001', 'nt_v2_500m_dna_1001', 'esm1b_650m_prot_501', 'esm1v_650m_prot_501', 'esm2_650m_prot_501', 'esmc_600m_prot_501']
Splits: ['train3', 'val', 'test', 'clinvarhq', 'proteingym', 'uniprot']


### Ki?m k? cache tr??c khi tr?ch xu?t
Cell ch? ??c manifest/t?n file; kh?ng n?p FM v? kh?ng ch?y inference. Pooled cache ???c ki?m tra cho ?? d?i ??u v?o b??c 1 v? to?n b? sequence-length ?ng vi?n b??c 2. Token cache ???c ki?m tra cho b??c 1.


In [3]:
# Inventory pooled vectors needed by steps 1 and 2; pooling ablation itself remains deferred to step 3.
DNA_LENGTHS = (301, 1001, 2001, 10001)
PROTEIN_LENGTHS = (31, 101, 201, 501, 1001)
SPLIT_NAMES = [s["name"] for s in STEP1_SPLITS]
pooled_rows = []
for split in SPLIT_NAMES:
    for base in BASE_MODELS:
        modality = "dna" if base["seq_type"] == "dna" else "prot"
        lengths = (1001,) if modality == "dna" else (501,)
        # Also inventory possible step-2 candidates; best model is chosen after step 1.
        lengths = DNA_LENGTHS if modality == "dna" else PROTEIN_LENGTHS
        for length in lengths:
            model_name = f"{base['name']}_{modality}_{length}"
            missing = [pool for pool in ("center", "mean", "cls")
                       if not (POOLED_CACHE_ROOT / split / f"{model_name}_{pool}.pt").is_file()]
            pooled_rows.append({"Split": split, "Model": model_name,
                                "Pooling_Cache": "complete" if not missing else "missing: " + ", ".join(missing)})
POOLED_CACHE_INVENTORY = pd.DataFrame(pooled_rows)
TOKEN_CACHE_INVENTORY = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, STEP1_TOKEN_MODELS, STEP1_SPLITS))
display(POOLED_CACHE_INVENTORY)
display(TOKEN_CACHE_INVENTORY[["Model", "Split", "Status", "Shards", "Path"]])
print("Pooled model/split entries missing at least one pooling:", int((POOLED_CACHE_INVENTORY.Pooling_Cache != "complete").sum()))
print("Token model/split entries not complete:", int((TOKEN_CACHE_INVENTORY.Status != "complete").sum()))


,Split,Model,Pooling_Cache
0,train3,nt_v1_500m_dna_301,complete
1,train3,nt_v1_500m_dna_1001,complete
2,train3,nt_v1_500m_dna_2001,complete
3,train3,nt_v1_500m_dna_10001,"missing: center, mean, cls"
4,train3,nt_v3_650m_dna_301,complete
...,...,...,...
187,uniprot,esmc_600m_prot_31,complete
188,uniprot,esmc_600m_prot_101,complete
189,uniprot,esmc_600m_prot_201,complete
190,uniprot,esmc_600m_prot_501,complete


,Model,Split,Status,Shards,Path
0,nt_v1_500m_dna_1001,train3,complete,11604,F:\variant_data\fm_embeddings\token_level_cach...
1,nt_v1_500m_dna_1001,val,complete,6851,F:\variant_data\fm_embeddings\token_level_cach...
2,nt_v1_500m_dna_1001,test,complete,3048,F:\variant_data\fm_embeddings\token_level_cach...
3,nt_v1_500m_dna_1001,clinvarhq,complete,352,F:\variant_data\fm_embeddings\token_level_cach...
4,nt_v1_500m_dna_1001,proteingym,complete,736,F:\variant_data\fm_embeddings\token_level_cach...
5,nt_v1_500m_dna_1001,uniprot,complete,667,F:\variant_data\fm_embeddings\token_level_cach...
6,nt_v3_650m_dna_1001,train3,complete,1451,F:\variant_data\fm_embeddings\token_level_cach...
7,nt_v3_650m_dna_1001,val,complete,857,F:\variant_data\fm_embeddings\token_level_cach...
8,nt_v3_650m_dna_1001,test,complete,381,F:\variant_data\fm_embeddings\token_level_cach...
9,nt_v3_650m_dna_1001,clinvarhq,complete,44,F:\variant_data\fm_embeddings\token_level_cach...


Pooled model/split entries missing at least one pooling: 6
Token model/split entries not complete: 0


### Tr?ch xu?t token-level c?n thi?u cho b??c 1
Cell b?t Hugging Face offline mode: n?u model weights ch?a c? trong cache c?c b?, extraction s? d?ng v?i l?i thay v? t?i ng?m. Cache ghi shard v? resume theo split. Ch?y xong cell n?y m?i ?? token cache ?? train 12 t? h?p.


In [4]:
# This cell performs local inference only. It does not train models or extract step-3 pooled features.
# HF_HUB_OFFLINE prevents accidental network downloads; ensure all seven model weights are cached locally.
os.environ["HF_HUB_OFFLINE"] = "1"
TOKEN_CACHE_PREPARATION = extract_missing_token_caches(
    model_configs=STEP1_TOKEN_MODELS,
    split_configs=STEP1_SPLITS,
    cache_root=TOKEN_CACHE_ROOT,
    batch_run_dir=FM_ABLATION_BATCH_RUN_DIR,
    device="cuda",
)
print("Token cache preparation:", TOKEN_CACHE_PREPARATION["status"])
print("Models newly processed/resumed:", TOKEN_CACHE_PREPARATION["extracted_models"])
TOKEN_CACHE_INVENTORY_AFTER = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, STEP1_TOKEN_MODELS, STEP1_SPLITS))
display(TOKEN_CACHE_INVENTORY_AFTER[["Model", "Split", "Status", "Shards"]])
if (TOKEN_CACHE_INVENTORY_AFTER.Status != "complete").any():
    raise RuntimeError("Token cache preparation incomplete; inspect statuses before starting ablation training.")


c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token cache preparation: complete
Models newly processed/resumed: []


,Model,Split,Status,Shards
0,nt_v1_500m_dna_1001,train3,complete,11604
1,nt_v1_500m_dna_1001,val,complete,6851
2,nt_v1_500m_dna_1001,test,complete,3048
3,nt_v1_500m_dna_1001,clinvarhq,complete,352
4,nt_v1_500m_dna_1001,proteingym,complete,736
5,nt_v1_500m_dna_1001,uniprot,complete,667
6,nt_v3_650m_dna_1001,train3,complete,1451
7,nt_v3_650m_dna_1001,val,complete,857
8,nt_v3_650m_dna_1001,test,complete,381
9,nt_v3_650m_dna_1001,clinvarhq,complete,44


### Step 1 ? run the 12 FM pairs
Run only after token-cache preparation is complete. The cell loads the selected Cross-Attention 3-modality Optuna best parameters from the configured source batch, fixes seed 42, and trains each DNA x protein pair on B_with_geometry. Resume/checkpoints and test metrics use the shared Module 6 runner.

For incomplete pairs, the first run creates reusable 33-token window shards next to the original token cache and loads the required windows into RAM. Preparation can take time and resumes at missing shards after interruption. Completed pairs are skipped; existing checkpoints retain their saved batch size. New pairs use batch size 64. Epoch timing includes validation and checkpoint writes.


In [5]:
# Fixed-parameter ablation: no new Optuna search in this step.
import importlib
import optuna
import torch
from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
architecture_runner_module = importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.architecture_experiment_runner"
))
run_architecture_experiment = architecture_runner_module.run_architecture_experiment
from core.module06_architecture_experiments.cross_attention_3modal_experiment import (
    DEFAULT_CROSS_ATTENTION_3MODAL_CONFIG, _builder as build_cross_attention_3modal,
)

# Source study containing the selected Cross-Attention 3-modality hyperparameters.
SOURCE_ARCH_BATCH_DIR = Path(r"D:\Missense_variant_pathogenicity_prediction\experiments\module06_architecture_run_20261104_071811")
SOURCE_STUDY_KEY = "cross_attention_3modal_seed_42"
source_study_db = SOURCE_ARCH_BATCH_DIR / "module06_architecture_experiments" / SOURCE_STUDY_KEY / "optuna" / "optuna_studies.sqlite3"
if not source_study_db.is_file():
    raise FileNotFoundError(f"Selected Cross-Attention Optuna study not found: {source_study_db}")
study = optuna.load_study(
    study_name=f"module06_{SOURCE_STUDY_KEY}",
    storage=f"sqlite:///{source_study_db.resolve().as_posix()}",
)
CROSS_ATTENTION_FIXED_CONFIG = {**DEFAULT_CROSS_ATTENTION_3MODAL_CONFIG, **study.best_params}
CROSS_ATTENTION_FIXED_CONFIG.update({"seed": 42, "k_radius": 16})
STEP1_SEEDS = [42, 37, 73, 97, 17]
print("Using fixed study params:", CROSS_ATTENTION_FIXED_CONFIG)

GEOMETRY_K = 64
GEOM_FEATURE_NAMES = [f"{m}_{f}" for m in ("dna", "prot") for f in ("LVD_L2", "LVD_Cosine", "LID", "LVD_Relative")]
TARGET_CONFIGS = [{
    "name": "B_with_geometry",
    "ablation": "bio_core_dna_prot_spliceai_geom",
    "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
    "geom_feature_names": GEOM_FEATURE_NAMES,
    # Geometry files remain fixed to the baseline FM pair while FM embeddings vary.
    "geometry_dna_name": "nt_v1_500m_dna_1001",
    "geometry_protein_name": "esm2_650m_prot_501",
}]
DATASETS = [
    {"name": "Train3Val_Test", "train": "train3", "val": "val", "test": "test"},
    {"name": "Train3Val_ClinVarHQ", "train": "train3", "val": "val", "test": "clinvarhq"},
    {"name": "Train3Val_ProteinGym", "train": "train3", "val": "val", "test": "proteingym"},
    {"name": "Train3Val_UniProt", "train": "train3", "val": "val", "test": "uniprot"},
]
PIPELINE_CONFIG = {"batch_size": 256, "epochs": 30, "lr": 1e-4, "weight_decay": 1e-3,
                   "lr_factor": 0.5, "lr_patience": 3, "early_stop_patience": 6,
                   "num_workers": 0, "seed": 42, "xgb_n_jobs": 8}
CROSS_ATTENTION_3MODAL_BATCH_SIZE = 64
LEGACY_RESUME_TOKEN_BATCH_SIZE = 64

# Stop before training if any requested token cache is absent/incomplete.
if "TOKEN_CACHE_PREPARATION" not in globals() or TOKEN_CACHE_PREPARATION.get("status") != "complete":
    raise RuntimeError("Run the token-cache preflight/extraction cell first.")
token_index = TOKEN_CACHE_PREPARATION["index"]
if any(token_index["models"].get(m["name"], {}).get(s["name"], {}).get("status") != "complete"
       for m in STEP1_TOKEN_MODELS for s in STEP1_SPLITS):
    raise RuntimeError("At least one FM token cache is incomplete; finish extraction before training.")

import json
PROFILE_METRIC_COLUMNS = architecture_runner_module.PROFILE_METRIC_COLUMNS
MANIFEST_ARCHITECTURE = "cross_attention_3modal_ref_alt_concat_window33_fm_pair_ablation"
PAIR_VARIANT = "ref_alt_concat_window_33_fixed_hparams"

def _load_pair_resume_settings(stage_dir, pair_key, dna_model, protein_model):
    manifest_path = stage_dir / "_resume_manifest.json"
    if not manifest_path.is_file():
        if stage_dir.exists() and any(path.is_file() for path in stage_dir.rglob("*")):
            raise ValueError(f"Existing stage artifacts have no resume manifest: {stage_dir}")
        return None

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    saved = manifest.get("config", {})
    expected = {
        "architecture": MANIFEST_ARCHITECTURE,
        "target_configs": TARGET_CONFIGS,
        "datasets": DATASETS,
        "dna_model": dna_model,
        "protein_model": protein_model,
        "pooling": "center",
        "embedding_representation": "ref_alt_concat",
        "checkpoint_selection": "validation_mcc_optimized_threshold_v2",
        "variant": PAIR_VARIANT,
    }
    mismatches = [key for key, value in expected.items() if saved.get(key) != value]
    if manifest.get("stage") != f"module06_{pair_key}" or mismatches:
        raise ValueError(
            f"Saved stage {stage_dir} is for different settings; mismatched fields: {mismatches}. "
            "Use a new batch run instead of reusing these artifacts."
        )
    config_key = f"{pair_key}_config"
    if config_key not in saved or "pipeline_config" not in saved:
        raise ValueError(f"Saved stage configuration is incomplete: {manifest_path}")
    return saved

def _completed_pair_metrics(stage_dir, pair_key):
    metrics_path = stage_dir / f"{pair_key}_test_metrics.csv"
    if not metrics_path.is_file():
        return None
    metrics = pd.read_csv(metrics_path)
    for target in TARGET_CONFIGS:
        for dataset in DATASETS:
            rows = metrics[
                (metrics["Target_Config"].astype(str) == target["name"])
                & (metrics["Dataset"].astype(str) == dataset["name"])
            ]
            if rows.empty:
                return None
            row = rows.iloc[0]
            if any(column not in rows.columns or pd.isna(row[column])
                   for column in PROFILE_METRIC_COLUMNS):
                return None
    return metrics

all_results = []
stage_root = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments"
for dna_model in [m for m in STEP1_TOKEN_MODELS if m["seq_type"] == "dna"]:
    for protein_model in [m for m in STEP1_TOKEN_MODELS if m["seq_type"] == "protein"]:
        for seed in STEP1_SEEDS:
            pair_base_key = f"step1_{dna_model['name']}__{protein_model['name']}"
            pair_key = pair_base_key if seed == 42 else f"{pair_base_key}_seed_{seed}"
            stage_dir = stage_root / pair_key
            saved_settings = _load_pair_resume_settings(stage_dir, pair_key, dna_model, protein_model)
            completed_metrics = _completed_pair_metrics(stage_dir, pair_key) if saved_settings else None
            if completed_metrics is not None:
                print(f"[Skip] Completed model pair: {pair_key}")
                all_results.append({"test_metrics": completed_metrics})
                continue

            if saved_settings:
                pair_pipeline_config = saved_settings["pipeline_config"]
                pair_experiment_config = saved_settings[f"{pair_key}_config"]
                saved_input_data_signature = saved_settings.get("input_data_signature")
                if saved_input_data_signature is None:
                    pair_token_batch_size = LEGACY_RESUME_TOKEN_BATCH_SIZE
                    pair_input_data_signature = None
                else:
                    pair_token_batch_size = int(
                        saved_input_data_signature.get(
                            "token_batch_size", LEGACY_RESUME_TOKEN_BATCH_SIZE
                        )
                    )
                    pair_input_data_signature = saved_input_data_signature
                print(f"[Resume] {pair_key} with its saved configuration")
            else:
                pair_pipeline_config = {**PIPELINE_CONFIG, "seed": seed}
                pair_experiment_config = {**CROSS_ATTENTION_FIXED_CONFIG, "seed": seed}
                pair_token_batch_size = CROSS_ATTENTION_3MODAL_BATCH_SIZE
                pair_input_data_signature = {"token_batch_size": pair_token_batch_size}
                print(f"[New] {pair_key} with the current configuration")
            print(f"[Train config] token_batch_size={pair_token_batch_size}")

            pipeline = FusionBatchPipeline(
                base_dir=str(BASE_DIR), config=pair_pipeline_config, datasets=DATASETS,
                models_space=[dna_model, protein_model], pooling_strategies=["center"],
                experiments=[{"name": "PyTorch_CrossAttention", "type": "pytorch", "fusion": "cross_attention"}],
                explainability={"enable_shap": False, "enable_lime": False},
                fm_profile_json=str(BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"),
                geom_profile_json=None,
                geometry_dir=str(BASE_DIR / f"geometry_v2_k{GEOMETRY_K}" / "{context}"),
                geom_feature_names=GEOM_FEATURE_NAMES,
                batch_run_dir=str(FM_ABLATION_BATCH_RUN_DIR),
                embedding_dir=str(POOLED_CACHE_ROOT),
            )
            context = {
                "pipeline": pipeline, "TARGET_CONFIGS": TARGET_CONFIGS, "datasets": DATASETS,
                "DNA_MODEL": dna_model, "PROTEIN_MODEL": protein_model,
                "DEVICE": pipeline.device, "token_embedding_cache_index": token_index,
                "token_batch_size": pair_token_batch_size,
            }
            if pair_input_data_signature is not None:
                context["input_data_signature"] = pair_input_data_signature
            print(f"\n[Step 1] {pair_key}")
            result = run_architecture_experiment(
                context=context, experiment_config=pair_experiment_config,
                model_builder=build_cross_attention_3modal,
                architecture="CrossAttention_ThreeModalities_FM_Pair_Ablation",
                variant=PAIR_VARIANT,
                experiment_key=pair_key, source="fm_ablation_step1",
                manifest_architecture=MANIFEST_ARCHITECTURE,
                embedding_representation="ref_alt_concat",
            )
            all_results.append(result)

STEP1_TEST_RESULTS = pd.concat(
    [item["test_metrics"] for item in all_results if "test_metrics" in item],
    ignore_index=True, sort=False,
) if all_results else pd.DataFrame()
if not STEP1_TEST_RESULTS.empty:
    display(STEP1_TEST_RESULTS.sort_values(["Dataset", "MCC"], ascending=[True, False]))


Using fixed study params: {'k_radius': 16, 'model_dim': 512, 'num_heads': 4, 'num_layers': 3, 'ffn_multiplier': 4, 'pooling': 'mean', 'dropout': 0.27733010051969553, 'attention_dropout': 0.09245839484800474, 'feature_dropout': 0.21563057108849548, 'max_epochs': 30, 'learning_rate': 1.0689751565977947e-05, 'weight_decay': 0.00039841905944346893, 'early_stop_patience': 6, 'lr_factor': 0.5, 'lr_patience': 3, 'seed': 42}
[Skip] Completed model pair: step1_nt_v1_500m_dna_1001__esm1b_650m_prot_501
[Skip] Completed model pair: step1_nt_v1_500m_dna_1001__esm1b_650m_prot_501_seed_37
[Skip] Completed model pair: step1_nt_v1_500m_dna_1001__esm1b_650m_prot_501_seed_73
[Skip] Completed model pair: step1_nt_v1_500m_dna_1001__esm1b_650m_prot_501_seed_97
[Skip] Completed model pair: step1_nt_v1_500m_dna_1001__esm1b_650m_prot_501_seed_17
[Skip] Completed model pair: step1_nt_v1_500m_dna_1001__esm1v_650m_prot_501
[Skip] Completed model pair: step1_nt_v1_500m_dna_1001__esm1v_650m_prot_501_seed_37
[Skip] 

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Classifier_GFLOPs/Sample,Classifier_Latency_ms/Sample,Classifier_Total_Params,Classifier_Param_Mem_MB,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB
13,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm1b_650m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3669,0.15,33273860,126.93,641.87,820.8911,35.54,1171332260,2297.61,2132.94
69,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esmc_600m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3583,0.15,33142788,126.43,640.27,744.2998,32.97,1093879086,2149.63,1546.25
205,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_ClinVarHQ,center,nt_v2_500m_dna_1001,esm2_650m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3496,0.14,33011716,125.93,632.13,825.3513,40.32,1182400406,2318.22,1790.91
61,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esmc_600m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3583,0.34,33142788,126.43,813.76,744.2998,33.16,1093879086,2149.63,1546.25
77,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esmc_600m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3583,0.14,33142788,126.43,635.94,744.2998,32.96,1093879086,2149.63,1546.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v2_500m_dna_1001,esm1v_650m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3496,0.14,33011716,125.93,638.41,825.3513,36.61,1183713686,2320.72,1793.29
115,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v3_650m_dna_1001,esm1v_650m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3842,0.14,33536004,127.93,645.84,779.2230,39.81,1337721973,3858.73,3427.19
107,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v3_650m_dna_1001,esm1v_650m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3842,0.15,33536004,127.93,648.37,779.2230,39.82,1337721973,3858.73,3427.19
231,fm_ablation_20261117_003149,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v2_500m_dna_1001,esmc_600m_prot_501,CrossAttention_ThreeModalities_FM_Pair_Ablation,...,1.3410,0.14,32880644,125.43,630.19,748.7652,35.65,1106263072,2172.75,1546.25


### Step 1 ? summarize the 12 FM pairs
This cell reads only completed Step 1 CSVs from the active FM ablation batch. Pair selection is ranked by validation MCC; test tables are shown separately for each dataset and are reporting-only.


In [6]:
from pathlib import Path
import pandas as pd
from IPython.display import display

STEP1_RESULT_ROOT = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments"
PAIR_KEYS = ["DNA_Model", "Prot_Model"]
validation_frames, test_frames = [], []
for path in sorted(STEP1_RESULT_ROOT.glob("step1_*/*_validation_metrics.csv")):
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        print(f"Skip unreadable validation file {path}: {exc}")
        continue
    if not frame.empty and "Source" in frame and frame["Source"].astype(str).eq("fm_ablation_step1").any():
        validation_frames.append(frame)
for path in sorted(STEP1_RESULT_ROOT.glob("step1_*/*_test_metrics.csv")):
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        print(f"Skip unreadable test file {path}: {exc}")
        continue
    if not frame.empty and "Source" in frame and frame["Source"].astype(str).eq("fm_ablation_step1").any():
        test_frames.append(frame)
if not validation_frames:
    raise FileNotFoundError(f"No Step 1 validation CSVs found in {STEP1_RESULT_ROOT}")

val = pd.concat(validation_frames, ignore_index=True, sort=False)
test = pd.concat(test_frames, ignore_index=True, sort=False) if test_frames else pd.DataFrame()

def _numeric_metrics(frame, excluded):
    return [c for c in frame.select_dtypes(include="number").columns if c not in excluded]

def _mean_std_by(frame, keys, excluded):
    metrics = _numeric_metrics(frame, excluded | {"N_Seeds"})
    grouped = frame.groupby(keys, dropna=False)[metrics].agg(["mean", "std"])
    grouped.columns = [f"{metric}_{stat}" for metric, stat in grouped.columns]
    counts = frame.groupby(keys, dropna=False)["Seed"].nunique().rename("N_Seeds")
    return grouped.join(counts).reset_index()

# Validation selection summary, shown separately from test reporting.
val_by_seed = val.drop_duplicates([*PAIR_KEYS, "Seed"])
val_summary = _mean_std_by(val_by_seed, PAIR_KEYS, set(PAIR_KEYS) | {"Seed"})
val_summary = val_summary.sort_values("Val_MCC_mean", ascending=False).reset_index(drop=True)
val_summary.insert(0, "Validation_Rank", range(1, len(val_summary) + 1))
print(f"Validation pairs: {len(val_summary)}/12; seeds observed: {sorted(val['Seed'].dropna().unique().tolist())}")
print("Validation metrics (mean/std across available seeds):")
display(val_summary)

# Each test dataset gets its own table, with every numeric metric/profile field summarized across seeds.
if not test.empty:
    test_by_seed = test.drop_duplicates([*PAIR_KEYS, "Dataset", "Seed"])
    test_dataset_summary = _mean_std_by(
        test_by_seed, [*PAIR_KEYS, "Dataset"], set(PAIR_KEYS) | {"Dataset", "Seed"}
    )
    for dataset_name, frame in test_dataset_summary.groupby("Dataset", sort=False):
        print(f"\nTest dataset: {dataset_name} ? metrics mean/std across seeds")
        display(frame.sort_values("MCC_mean", ascending=False).reset_index(drop=True))
else:
    test_dataset_summary = pd.DataFrame()
    print("No completed Step 1 test metrics found yet.")

# Save the validation summary and separate test-dataset summaries beside the source results.
val_summary.to_csv(STEP1_RESULT_ROOT / "step1_validation_metrics_mean_std_by_pair.csv", index=False)
if not test_dataset_summary.empty:
    test_dataset_summary.to_csv(STEP1_RESULT_ROOT / "step1_test_metrics_mean_std_by_pair_dataset.csv", index=False)
print("Summary CSVs saved under:", STEP1_RESULT_ROOT)


Validation pairs: 12/12; seeds observed: [17, 37, 42, 73, 97]
Validation metrics (mean/std across available seeds):


,Validation_Rank,DNA_Model,Prot_Model,Best_Epoch_mean,Best_Epoch_std,Fit_Seconds_mean,Fit_Seconds_std,Decision_Threshold_mean,Decision_Threshold_std,Val_MCC_mean,...,Val_Recall_std,Val_Specificity_mean,Val_Specificity_std,Val_F1_Score_mean,Val_F1_Score_std,Val_AUROC_mean,Val_AUROC_std,Val_AUPRC_mean,Val_AUPRC_std,N_Seeds
0,1,nt_v1_500m_dna_1001,esmc_600m_prot_501,24.2,2.863564,2181.616999,3967.815177,0.883536,0.055268,0.81706,...,0.007583,0.96480,0.002130,0.85198,0.001607,0.97412,0.000540,0.91612,0.003706,5
1,2,nt_v3_650m_dna_1001,esmc_600m_prot_501,21.4,10.807405,360.379046,129.829765,0.445243,0.169232,0.81362,...,0.024895,0.96650,0.007195,0.84862,0.017121,0.97352,0.005589,0.91798,0.015496,5
2,3,nt_v1_500m_dna_1001,esm1b_650m_prot_501,8.8,1.303840,1049.791772,1873.499419,0.771237,0.113445,0.81102,...,0.009930,0.96274,0.003335,0.84718,0.004164,0.97296,0.001234,0.91442,0.002137,5
3,4,nt_v1_500m_dna_1001,esm2_650m_prot_501,10.2,1.643168,1109.866303,1978.438210,0.825912,0.155170,0.81062,...,0.008545,0.96436,0.002402,0.84664,0.002897,0.97214,0.000934,0.91160,0.003188,5
4,5,nt_v3_650m_dna_1001,esm2_650m_prot_501,11.8,2.774887,7096.736015,15284.791538,0.481768,0.089722,0.80872,...,0.017518,0.96430,0.003920,0.84498,0.003604,0.97246,0.001062,0.91284,0.002767,5
5,6,nt_v1_500m_dna_1001,esm1v_650m_prot_501,9.2,1.643168,1035.424892,1813.461382,0.844422,0.066199,0.80638,...,0.010968,0.96498,0.002385,0.84298,0.004553,0.97158,0.000766,0.91184,0.002163,5
6,7,nt_v3_650m_dna_1001,esm1b_650m_prot_501,7.6,1.140175,5800.055115,12516.068479,0.609229,0.192683,0.80596,...,0.019805,0.96920,0.005605,0.84154,0.003879,0.97112,0.001190,0.91148,0.002588,5
7,8,nt_v2_500m_dna_1001,esmc_600m_prot_501,20.8,7.529940,359.172275,83.981539,0.864895,0.033134,0.80536,...,0.014449,0.96958,0.007910,0.84098,0.013159,0.97160,0.003421,0.91222,0.011698,5
8,9,nt_v2_500m_dna_1001,esm1b_650m_prot_501,10.2,2.280351,231.778542,30.880789,0.798375,0.181999,0.80496,...,0.015583,0.96574,0.004744,0.84154,0.002569,0.97186,0.000913,0.91066,0.003354,5
9,10,nt_v2_500m_dna_1001,esm2_650m_prot_501,11.2,2.683282,243.161040,37.955753,0.859718,0.071462,0.80308,...,0.020072,0.96142,0.004805,0.84058,0.004489,0.97112,0.001052,0.90558,0.002745,5



Test dataset: Train3Val_ClinVarHQ ? metrics mean/std across seeds


,DNA_Model,Prot_Model,Dataset,Decision_Threshold_mean,Decision_Threshold_std,Best_Epoch_mean,Best_Epoch_std,Fit_Seconds_mean,Fit_Seconds_std,Val_MCC_mean,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
0,nt_v1_500m_dna_1001,esmc_600m_prot_501,Train3Val_ClinVarHQ,0.883536,0.055268,24.2,2.863564,2181.616999,3967.815177,0.81706,...,0.0,33.004,0.087350,1.093879e+09,0.0,2149.63,0.0,1546.25,0.0,5
1,nt_v1_500m_dna_1001,esm1b_650m_prot_501,Train3Val_ClinVarHQ,0.771237,0.113445,8.8,1.303840,1049.791772,1873.499419,0.81102,...,0.0,35.582,0.083187,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,5
2,nt_v2_500m_dna_1001,esm2_650m_prot_501,Train3Val_ClinVarHQ,0.859718,0.071462,11.2,2.683282,243.161040,37.955753,0.80308,...,0.0,40.322,0.004472,1.182400e+09,0.0,2318.22,0.0,1790.91,0.0,5
3,nt_v1_500m_dna_1001,esm2_650m_prot_501,Train3Val_ClinVarHQ,0.825912,0.155170,10.2,1.643168,1109.866303,1978.438210,0.81062,...,0.0,38.396,0.170088,1.170016e+09,0.0,2295.10,0.0,1790.91,0.0,5
4,nt_v3_650m_dna_1001,esm2_650m_prot_501,Train3Val_ClinVarHQ,0.481768,0.089722,11.8,2.774887,7096.736015,15284.791538,0.80872,...,0.0,43.684,0.154370,1.336409e+09,0.0,3856.23,0.0,3427.19,0.0,5
5,nt_v2_500m_dna_1001,esm1v_650m_prot_501,Train3Val_ClinVarHQ,0.863119,0.093965,10.4,2.073644,233.088380,27.873714,0.80116,...,0.0,36.866,0.013416,1.183714e+09,0.0,2320.72,0.0,1793.29,0.0,5
6,nt_v2_500m_dna_1001,esm1b_650m_prot_501,Train3Val_ClinVarHQ,0.798375,0.181999,10.2,2.280351,231.778542,30.880789,0.80496,...,0.0,37.538,0.008367,1.183716e+09,0.0,2320.73,0.0,2132.94,0.0,5
7,nt_v3_650m_dna_1001,esmc_600m_prot_501,Train3Val_ClinVarHQ,0.445243,0.169232,21.4,10.807405,360.379046,129.829765,0.81362,...,0.0,38.258,0.014832,1.260271e+09,0.0,3710.76,0.0,3427.19,0.0,5
8,nt_v2_500m_dna_1001,esmc_600m_prot_501,Train3Val_ClinVarHQ,0.864895,0.033134,20.8,7.529940,359.172275,83.981539,0.80536,...,0.0,34.960,0.000000,1.106263e+09,0.0,2172.75,0.0,1546.25,0.0,5
9,nt_v1_500m_dna_1001,esm1v_650m_prot_501,Train3Val_ClinVarHQ,0.844422,0.066199,9.2,1.643168,1035.424892,1813.461382,0.80638,...,0.0,34.926,0.130882,1.171330e+09,0.0,2297.60,0.0,1793.29,0.0,5



Test dataset: Train3Val_ProteinGym ? metrics mean/std across seeds


,DNA_Model,Prot_Model,Dataset,Decision_Threshold_mean,Decision_Threshold_std,Best_Epoch_mean,Best_Epoch_std,Fit_Seconds_mean,Fit_Seconds_std,Val_MCC_mean,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
0,nt_v1_500m_dna_1001,esm1b_650m_prot_501,Train3Val_ProteinGym,0.771237,0.113445,8.8,1.303840,1049.791772,1873.499419,0.81102,...,0.0,35.100,0.114673,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,5
1,nt_v2_500m_dna_1001,esm1b_650m_prot_501,Train3Val_ProteinGym,0.798375,0.181999,10.2,2.280351,231.778542,30.880789,0.80496,...,0.0,37.006,0.008944,1.183716e+09,0.0,2320.73,0.0,2132.94,0.0,5
2,nt_v1_500m_dna_1001,esmc_600m_prot_501,Train3Val_ProteinGym,0.883536,0.055268,24.2,2.863564,2181.616999,3967.815177,0.81706,...,0.0,33.732,0.116276,1.093879e+09,0.0,2149.63,0.0,1546.25,0.0,5
3,nt_v2_500m_dna_1001,esmc_600m_prot_501,Train3Val_ProteinGym,0.864895,0.033134,20.8,7.529940,359.172275,83.981539,0.80536,...,0.0,35.670,0.000000,1.106263e+09,0.0,2172.75,0.0,1546.25,0.0,5
4,nt_v1_500m_dna_1001,esm2_650m_prot_501,Train3Val_ProteinGym,0.825912,0.155170,10.2,1.643168,1109.866303,1978.438210,0.81062,...,0.0,38.156,0.108536,1.170016e+09,0.0,2295.10,0.0,1790.91,0.0,5
5,nt_v3_650m_dna_1001,esm1b_650m_prot_501,Train3Val_ProteinGym,0.609229,0.192683,7.6,1.140175,5800.055115,12516.068479,0.80596,...,0.0,40.384,0.232551,1.337725e+09,0.0,3858.74,0.0,3427.19,0.0,5
6,nt_v1_500m_dna_1001,esm1v_650m_prot_501,Train3Val_ProteinGym,0.844422,0.066199,9.2,1.643168,1035.424892,1813.461382,0.80638,...,0.0,34.696,0.108536,1.171330e+09,0.0,2297.60,0.0,1793.29,0.0,5
7,nt_v3_650m_dna_1001,esm1v_650m_prot_501,Train3Val_ProteinGym,0.550343,0.034205,13.2,5.449771,6624.141914,14142.269334,0.80020,...,0.0,40.044,0.288496,1.337722e+09,0.0,3858.73,0.0,3427.19,0.0,5
8,nt_v2_500m_dna_1001,esm2_650m_prot_501,Train3Val_ProteinGym,0.859718,0.071462,11.2,2.683282,243.161040,37.955753,0.80308,...,0.0,40.096,0.008944,1.182400e+09,0.0,2318.22,0.0,1790.91,0.0,5
9,nt_v2_500m_dna_1001,esm1v_650m_prot_501,Train3Val_ProteinGym,0.863119,0.093965,10.4,2.073644,233.088380,27.873714,0.80116,...,0.0,36.634,0.008944,1.183714e+09,0.0,2320.72,0.0,1793.29,0.0,5



Test dataset: Train3Val_Test ? metrics mean/std across seeds


,DNA_Model,Prot_Model,Dataset,Decision_Threshold_mean,Decision_Threshold_std,Best_Epoch_mean,Best_Epoch_std,Fit_Seconds_mean,Fit_Seconds_std,Val_MCC_mean,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
0,nt_v1_500m_dna_1001,esmc_600m_prot_501,Train3Val_Test,0.883536,0.055268,24.2,2.863564,2181.616999,3967.815177,0.81706,...,0.0,32.320,0.117473,1.093879e+09,0.0,2149.63,0.0,1546.25,0.0,5
1,nt_v1_500m_dna_1001,esm1b_650m_prot_501,Train3Val_Test,0.771237,0.113445,8.8,1.303840,1049.791772,1873.499419,0.81102,...,0.0,34.674,0.065422,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,5
2,nt_v2_500m_dna_1001,esmc_600m_prot_501,Train3Val_Test,0.864895,0.033134,20.8,7.529940,359.172275,83.981539,0.80536,...,0.0,34.180,0.000000,1.106263e+09,0.0,2172.75,0.0,1546.25,0.0,5
3,nt_v2_500m_dna_1001,esm1b_650m_prot_501,Train3Val_Test,0.798375,0.181999,10.2,2.280351,231.778542,30.880789,0.80496,...,0.0,36.556,0.008944,1.183716e+09,0.0,2320.73,0.0,2132.94,0.0,5
4,nt_v1_500m_dna_1001,esm2_650m_prot_501,Train3Val_Test,0.825912,0.155170,10.2,1.643168,1109.866303,1978.438210,0.81062,...,0.0,37.998,0.107331,1.170016e+09,0.0,2295.10,0.0,1790.91,0.0,5
5,nt_v3_650m_dna_1001,esmc_600m_prot_501,Train3Val_Test,0.445243,0.169232,21.4,10.807405,360.379046,129.829765,0.81362,...,0.0,37.470,0.007071,1.260271e+09,0.0,3710.76,0.0,3427.19,0.0,5
6,nt_v2_500m_dna_1001,esm2_650m_prot_501,Train3Val_Test,0.859718,0.071462,11.2,2.683282,243.161040,37.955753,0.80308,...,0.0,39.870,0.017321,1.182400e+09,0.0,2318.22,0.0,1790.91,0.0,5
7,nt_v3_650m_dna_1001,esm1b_650m_prot_501,Train3Val_Test,0.609229,0.192683,7.6,1.140175,5800.055115,12516.068479,0.80596,...,0.0,39.938,0.219135,1.337725e+09,0.0,3858.74,0.0,3427.19,0.0,5
8,nt_v2_500m_dna_1001,esm1v_650m_prot_501,Train3Val_Test,0.863119,0.093965,10.4,2.073644,233.088380,27.873714,0.80116,...,0.0,36.348,0.013038,1.183714e+09,0.0,2320.72,0.0,1793.29,0.0,5
9,nt_v1_500m_dna_1001,esm1v_650m_prot_501,Train3Val_Test,0.844422,0.066199,9.2,1.643168,1035.424892,1813.461382,0.80638,...,0.0,34.470,0.089443,1.171330e+09,0.0,2297.60,0.0,1793.29,0.0,5



Test dataset: Train3Val_UniProt ? metrics mean/std across seeds


,DNA_Model,Prot_Model,Dataset,Decision_Threshold_mean,Decision_Threshold_std,Best_Epoch_mean,Best_Epoch_std,Fit_Seconds_mean,Fit_Seconds_std,Val_MCC_mean,...,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std,N_Seeds
0,nt_v1_500m_dna_1001,esm1b_650m_prot_501,Train3Val_UniProt,0.771237,0.113445,8.8,1.303840,1049.791772,1873.499419,0.81102,...,0.0,35.158,0.113225,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0,5
1,nt_v2_500m_dna_1001,esm1b_650m_prot_501,Train3Val_UniProt,0.798375,0.181999,10.2,2.280351,231.778542,30.880789,0.80496,...,0.0,37.064,0.005477,1.183716e+09,0.0,2320.73,0.0,2132.94,0.0,5
2,nt_v1_500m_dna_1001,esmc_600m_prot_501,Train3Val_UniProt,0.883536,0.055268,24.2,2.863564,2181.616999,3967.815177,0.81706,...,0.0,33.740,0.106301,1.093879e+09,0.0,2149.63,0.0,1546.25,0.0,5
3,nt_v3_650m_dna_1001,esm1b_650m_prot_501,Train3Val_UniProt,0.609229,0.192683,7.6,1.140175,5800.055115,12516.068479,0.80596,...,0.0,40.370,0.229238,1.337725e+09,0.0,3858.74,0.0,3427.19,0.0,5
4,nt_v1_500m_dna_1001,esm1v_650m_prot_501,Train3Val_UniProt,0.844422,0.066199,9.2,1.643168,1035.424892,1813.461382,0.80638,...,0.0,34.696,0.091815,1.171330e+09,0.0,2297.60,0.0,1793.29,0.0,5
5,nt_v2_500m_dna_1001,esm1v_650m_prot_501,Train3Val_UniProt,0.863119,0.093965,10.4,2.073644,233.088380,27.873714,0.80116,...,0.0,36.608,0.010954,1.183714e+09,0.0,2320.72,0.0,1793.29,0.0,5
6,nt_v1_500m_dna_1001,esm2_650m_prot_501,Train3Val_UniProt,0.825912,0.155170,10.2,1.643168,1109.866303,1978.438210,0.81062,...,0.0,38.278,0.185661,1.170016e+09,0.0,2295.10,0.0,1790.91,0.0,5
7,nt_v2_500m_dna_1001,esm2_650m_prot_501,Train3Val_UniProt,0.859718,0.071462,11.2,2.683282,243.161040,37.955753,0.80308,...,0.0,40.152,0.004472,1.182400e+09,0.0,2318.22,0.0,1790.91,0.0,5
8,nt_v2_500m_dna_1001,esmc_600m_prot_501,Train3Val_UniProt,0.864895,0.033134,20.8,7.529940,359.172275,83.981539,0.80536,...,0.0,35.652,0.004472,1.106263e+09,0.0,2172.75,0.0,1546.25,0.0,5
9,nt_v3_650m_dna_1001,esmc_600m_prot_501,Train3Val_UniProt,0.445243,0.169232,21.4,10.807405,360.379046,129.829765,0.81362,...,0.0,38.862,0.026833,1.260271e+09,0.0,3710.76,0.0,3427.19,0.0,5


Summary CSVs saved under: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments


### Optuna fine-tuning ? best FM pair, shared hyperparameters across five seeds
The study optimizes mean validation MCC over seeds 17, 37, 42, 73, and 97. Test sets are not used by Optuna; the selected configuration is evaluated on the four test datasets for every seed afterward. Edit the search ranges and trial budget in this cell.


In [ ]:
import hashlib
import json
import optuna
import pandas as pd
from optuna.trial import TrialState
from IPython.display import display
from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module06_architecture_experiments.cross_attention_3modal_experiment import DEFAULT_CROSS_ATTENTION_3MODAL_SEARCH_SPACE

OPTUNA_DNA_MODEL = next(m for m in STEP1_TOKEN_MODELS if m["name"] == "nt_v1_500m_dna_1001")
OPTUNA_PROTEIN_MODEL = next(m for m in STEP1_TOKEN_MODELS if m["name"] == "esm1b_650m_prot_501")
OPTUNA_SEEDS = [17, 37, 42, 73, 97]
OPTUNA_TRIAL_BUDGET = 60  # Total completed trials; reruns add only missing trials.
OPTUNA_SEARCH_SPACE = {
    "num_heads": {"type": "categorical", "choices": [4, 8, 16]},
    "ffn_multiplier": {"type": "categorical", "choices": [4, 8, 16]},
    "dropout": {"type": "float", "low": 0.05, "high": 0.40},
    "attention_dropout": {"type": "float", "low": 0.005, "high": 0.40},
    "feature_dropout": {"type": "float", "low": 0.005, "high": 0.40},
    "learning_rate": {"type": "float", "low": 1e-6, "high": 2e-3, "log": True},
    "weight_decay": {"type": "float", "low": 1e-6, "high": 0.2, "log": True},
    # Có thể thêm hoặc bỏ tham số cần tìm ở đây.
}
OPTUNA_EXPERIMENT_KEY = "step1_optuna_nt_v1_dna1001_esm1b_prot501_5seed"
OPTUNA_STUDY_NAME = "fm_pair_nt_v1_esm1b_mean_val_mcc_5seed"
OPTUNA_ROOT = Path(FM_ABLATION_BATCH_RUN_DIR) / "module06_architecture_experiments" / OPTUNA_EXPERIMENT_KEY
OPTUNA_ROOT.mkdir(parents=True, exist_ok=True)
OPTUNA_DB = OPTUNA_ROOT / "optuna_studies.sqlite3"
OPTUNA_STORAGE = f"sqlite:///{OPTUNA_DB.resolve().as_posix()}"

if "TOKEN_CACHE_PREPARATION" not in globals() or TOKEN_CACHE_PREPARATION.get("status") != "complete":
    raise RuntimeError("Complete token-cache extraction before Optuna fine-tuning.")
OPTUNA_TOKEN_INDEX = TOKEN_CACHE_PREPARATION["index"]
for model in (OPTUNA_DNA_MODEL, OPTUNA_PROTEIN_MODEL):
    for split in STEP1_SPLITS:
        status = OPTUNA_TOKEN_INDEX["models"].get(model["name"], {}).get(split["name"], {}).get("status")
        if status != "complete":
            raise RuntimeError(f"Incomplete token cache: {model['name']}/{split['name']}")

FAST_WINDOW_BATCH_SIZE = 64
FIXED_OPTUNA_BASE_CONFIG = {**CROSS_ATTENTION_FIXED_CONFIG, "k_radius": 16}
study = optuna.create_study(
    study_name=OPTUNA_STUDY_NAME, storage=OPTUNA_STORAGE, direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42), load_if_exists=True,
)
expected_search_params = set(OPTUNA_SEARCH_SPACE)
# Requeue sampled params from a process interrupted while the Optuna trial was running.
for frozen_trial in study.get_trials(deepcopy=False, states=(TrialState.RUNNING,)):
    if expected_search_params.issubset(frozen_trial.params):
        study.enqueue_trial(frozen_trial.params,
                            user_attrs={"requeued_from_interrupted_trial": int(frozen_trial.number)},
                            skip_if_exists=False)
    study.tell(frozen_trial.number, state=TrialState.FAIL)


def _suggest_one(trial, name, spec):
    kind = spec["type"]
    if kind == "categorical":
        return trial.suggest_categorical(name, spec["choices"])
    if kind == "int":
        return trial.suggest_int(name, int(spec["low"]), int(spec["high"]),
                                 step=int(spec.get("step", 1)), log=bool(spec.get("log", False)))
    if kind == "float":
        return trial.suggest_float(name, float(spec["low"]), float(spec["high"]),
                                   step=spec.get("step"), log=bool(spec.get("log", False)))
    raise ValueError(f"Unsupported Optuna parameter type {kind!r} for {name}")


def _params_fingerprint(params):
    payload = json.dumps(params, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def _pipeline_for_seed(seed):
    return FusionBatchPipeline(
        base_dir=str(BASE_DIR), config={**PIPELINE_CONFIG, "seed": int(seed)}, datasets=DATASETS,
        models_space=[OPTUNA_DNA_MODEL, OPTUNA_PROTEIN_MODEL], pooling_strategies=["center"],
        experiments=[{"name": "PyTorch_CrossAttention", "type": "pytorch", "fusion": "cross_attention"}],
        explainability={"enable_shap": False, "enable_lime": False},
        fm_profile_json=str(BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"),
        geom_profile_json=None,
        geometry_dir=str(BASE_DIR / f"geometry_v2_k{GEOMETRY_K}" / "{context}"),
        geom_feature_names=GEOM_FEATURE_NAMES,
        batch_run_dir=str(FM_ABLATION_BATCH_RUN_DIR), embedding_dir=str(POOLED_CACHE_ROOT),
    )

OPTUNA_PIPELINES = {seed: _pipeline_for_seed(seed) for seed in OPTUNA_SEEDS}


def _run_seed_for_trial(params, seed, evaluate_test=False):
    config = {**FIXED_OPTUNA_BASE_CONFIG, **params, "seed": int(seed), "k_radius": 16}
    pipeline = OPTUNA_PIPELINES[seed]
    context = {
        "pipeline": pipeline, "TARGET_CONFIGS": TARGET_CONFIGS, "datasets": DATASETS,
        "DNA_MODEL": OPTUNA_DNA_MODEL, "PROTEIN_MODEL": OPTUNA_PROTEIN_MODEL,
        "DEVICE": pipeline.device, "token_embedding_cache_index": OPTUNA_TOKEN_INDEX,
        "token_batch_size": FAST_WINDOW_BATCH_SIZE,
        "input_data_signature": {"token_batch_size": FAST_WINDOW_BATCH_SIZE,
                                 "window_cache": "variant_centered_33_token_v1"},
    }
    trial_name = f"params_{_params_fingerprint(params)}_seed_{seed}"
    return run_architecture_experiment(
        context=context, experiment_config=config, model_builder=build_cross_attention_3modal,
        architecture="CrossAttention_ThreeModalities_FM_Pair_Optuna",
        variant=PAIR_VARIANT, experiment_key=OPTUNA_EXPERIMENT_KEY, trial_name=trial_name,
        source="fm_ablation_step1", manifest_architecture=MANIFEST_ARCHITECTURE,
        embedding_representation="ref_alt_concat", evaluate_test=evaluate_test,
    )


def _objective(trial):
    params = {name: _suggest_one(trial, name, spec) for name, spec in OPTUNA_SEARCH_SPACE.items()}
    trial.set_user_attr("shared_seed_list", OPTUNA_SEEDS)
    trial.set_user_attr("fm_pair", [OPTUNA_DNA_MODEL["name"], OPTUNA_PROTEIN_MODEL["name"]])
    per_seed = {}
    for seed in OPTUNA_SEEDS:
        result = _run_seed_for_trial(params, seed, evaluate_test=False)
        rows = result["validation_metrics"]
        row = rows[rows["Target_Config"].astype(str).eq("B_with_geometry")].iloc[0]
        per_seed[str(seed)] = {
            "Val_MCC": float(row["Val_MCC"]),
            "Val_AUROC": float(row.get("Val_AUROC", float("nan"))),
            "Val_AUPRC": float(row.get("Val_AUPRC", float("nan"))),
            "Decision_Threshold": float(row["Decision_Threshold"]),
            "Best_Epoch": int(row["Best_Epoch"]),
        }
        trial.set_user_attr("per_seed_validation", per_seed)
    mcc_values = [result["Val_MCC"] for result in per_seed.values()]
    mean_mcc = float(pd.Series(mcc_values).mean())
    trial.set_user_attr("mean_val_mcc_5_seeds", mean_mcc)
    trial.set_user_attr("std_val_mcc_5_seeds", float(pd.Series(mcc_values).std()))
    return mean_mcc

completed_count = sum(t.state == TrialState.COMPLETE for t in study.get_trials(deepcopy=False))
trials_remaining = max(0, int(OPTUNA_TRIAL_BUDGET) - completed_count)
if trials_remaining:
    study.optimize(_objective, n_trials=trials_remaining, catch=(Exception,))
if not any(t.state == TrialState.COMPLETE for t in study.get_trials(deepcopy=False)):
    raise RuntimeError("Optuna has no completed trials; inspect trial errors before proceeding.")

BEST_OPTUNA_TRIAL = study.best_trial
BEST_OPTUNA_PARAMS = dict(BEST_OPTUNA_TRIAL.params)
BEST_OPTUNA_CONFIG = {**FIXED_OPTUNA_BASE_CONFIG, **BEST_OPTUNA_PARAMS, "k_radius": 16}
print("Best mean validation MCC across five seeds:", study.best_value)
print("Best shared hyperparameters:", BEST_OPTUNA_CONFIG)

# Test evaluation is only for the selected hyperparameter configuration, never the Optuna objective.
final_seed_results = []
for seed in OPTUNA_SEEDS:
    result = _run_seed_for_trial(BEST_OPTUNA_PARAMS, seed, evaluate_test=True)
    val_rows = result["validation_metrics"].copy(); val_rows["Seed"] = seed
    test_rows = result["test_metrics"].copy(); test_rows["Seed"] = seed
    final_seed_results.append({"Validation": val_rows, "Test": test_rows})
OPTUNA_BEST_VALIDATION_BY_SEED = pd.concat([x["Validation"] for x in final_seed_results], ignore_index=True)
OPTUNA_BEST_TEST_BY_SEED = pd.concat([x["Test"] for x in final_seed_results], ignore_index=True)
val_mcc_by_seed = OPTUNA_BEST_VALIDATION_BY_SEED[["Seed", "Val_MCC", "Decision_Threshold", "Best_Epoch"]].drop_duplicates("Seed")
test_mcc_by_seed = OPTUNA_BEST_TEST_BY_SEED[["Seed", "Dataset", "MCC"]].drop_duplicates(["Seed", "Dataset"])
wide_test = test_mcc_by_seed.pivot(index="Seed", columns="Dataset", values="MCC").reset_index()
seed_mcc_table = val_mcc_by_seed.merge(wide_test, on="Seed", how="outer").sort_values("Seed")
summary_columns = [c for c in seed_mcc_table.select_dtypes(include="number").columns if c != "Seed"]
mean_row = {"Seed": "Mean", **{c: float(seed_mcc_table[c].mean()) for c in summary_columns}}
std_row = {"Seed": "Std", **{c: float(seed_mcc_table[c].std()) for c in summary_columns}}
seed_mcc_table = pd.concat([seed_mcc_table, pd.DataFrame([mean_row, std_row])], ignore_index=True)
print("Per-seed MCC plus mean/std for the same selected hyperparameters:")
display(seed_mcc_table)

test_target_summary = test_mcc_by_seed.groupby("Dataset")["MCC"].agg(["mean", "std", "count"]).reset_index()
test_target_summary["Target_MCC_gt_0_88"] = test_target_summary["mean"] > 0.88
print("MCC mean/std by test set across five seeds (reporting only):")
display(test_target_summary)
mean_val_mcc = float(val_mcc_by_seed["Val_MCC"].mean())
print(f"Validation target > 0.818: {mean_val_mcc > 0.818} (mean={mean_val_mcc:.4f})")
print("All four test-set mean MCC targets > 0.88:", bool(test_target_summary["Target_MCC_gt_0_88"].all()))

study.trials_dataframe(attrs=("number", "value", "params", "state", "user_attrs")).to_csv(OPTUNA_ROOT / "optuna_trials.csv", index=False)
(OPTUNA_ROOT / "best_shared_hyperparameters.json").write_text(
    json.dumps({"best_trial": BEST_OPTUNA_TRIAL.number, "mean_val_mcc_5_seeds": study.best_value,
                "seed_list": OPTUNA_SEEDS, "params": BEST_OPTUNA_CONFIG}, indent=2), encoding="utf-8")
OPTUNA_BEST_VALIDATION_BY_SEED.to_csv(OPTUNA_ROOT / "best_validation_metrics_by_seed.csv", index=False)
OPTUNA_BEST_TEST_BY_SEED.to_csv(OPTUNA_ROOT / "best_test_metrics_by_seed.csv", index=False)
seed_mcc_table.to_csv(OPTUNA_ROOT / "best_mcc_by_seed_mean_std.csv", index=False)
test_target_summary.to_csv(OPTUNA_ROOT / "best_test_mcc_mean_std_by_dataset.csv", index=False)
print("Optuna and five-seed outputs saved under:", OPTUNA_ROOT)


[I 2026-11-18 23:18:41,030] A new study created in RDB with name: fm_pair_nt_v1_esm1b_mean_val_mcc_5seed


[CrossAttention_ThreeModalities_FM_Pair_Optuna] Preparing train dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/train3: 100%|██████████| 11604/11604 [00:14<00:00, 805.09shard/s]


[Window cache] dna/train3: 23208 samples ready in 14.4s


Window cache protein/train3: 100%|██████████| 1451/1451 [00:01<00:00, 805.32shard/s]


[Window cache] protein/train3: 23208 samples ready in 1.8s
[CrossAttention_ThreeModalities_FM_Pair_Optuna] Train loader ready: 363 batches in 18.8s
[CrossAttention_ThreeModalities_FM_Pair_Optuna] Preparing validation dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai', 'geom']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



... (5,953 outputs left out)

CrossAttention_ThreeModalities_FM_Pair_Optuna | ref_alt_concat_window_33_fixed_hparams | B_with_geometry:  20%|██        | 6/30 [01:30<06:04, 15.21s/epoch, best_epoch=5, threshold=0.7413, train_loss=0.1651, val_MCC=0.7801, val_loss=0.2156]

[Epoch 6] train=11.7s; total including validation and checkpoint=15.2s


CrossAttention_ThreeModalities_FM_Pair_Optuna | ref_alt_concat_window_33_fixed_hparams | B_with_geometry:  23%|██▎       | 7/30 [01:45<05:49, 15.18s/epoch, best_epoch=5, threshold=0.8224, train_loss=0.1626, val_MCC=0.7949, val_loss=0.2163]

[Epoch 7] train=11.6s; total including validation and checkpoint=15.1s


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import optuna
from IPython.display import display

# This is the only required selection. Use exact Architecture names listed above.
MODEL_NAMES_TO_INSPECT = ["CrossAttention_ThreeModalities"]

# Optional: set a batch folder to inspect a historical run; leave None for automatic lookup.
BATCH_RUN_DIR_OVERRIDE = r"D:\Missense_variant_pathogenicity_prediction\experiments\module06_architecture_run_20261104_071811"

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (_cwd, *_cwd.parents)
     if (candidate / "core").is_dir()
     and (candidate / "controllers").is_dir()
     and (candidate / "experiments").is_dir()),
    _cwd,
)
EXPERIMENTS_ROOT = REPO_ROOT / "experiments"
if not MODEL_NAMES_TO_INSPECT:
    raise ValueError("Enter one or more architecture names in MODEL_NAMES_TO_INSPECT.")
